In [ ]:
from flask import Flask, render_template, request, jsonify
import pickle
import numpy as np

app = Flask(__name__)

# Load saved pickle artifacts on app startup
with open('diagonsis_detection.pkl', 'rb') as f:
    pipeline = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

@app.route('/')
def home():
    # Serves the HTML frontend page
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        
        # Extract features array sent from the frontend
        features = data.get('features')
        
        if not features or len(features) != 30:
            return jsonify({'error': 'Expected exactly 30 numerical features'}), 400

        # Reshape input for model prediction: shape (1, 30)
        input_data = np.array(features).reshape(1, -1)

        # Predict numeric label using the pipeline
        raw_pred = pipeline.predict(input_data)
        
        # Get prediction probabilities (optional, nice for UI)
        probabilities = pipeline.predict_proba(input_data)[0]

        # Decode numeric label back to original class string
        predicted_class = label_encoder.inverse_transform(raw_pred)[0]

        return jsonify({
            'status': 'success',
            'prediction': str(predicted_class),
            'confidence': float(np.max(probabilities))
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    # Run locally on port 5000
    app.run(debug=True, port=5000,use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
